# NB25: SpatiaLite Cache & Advanced Geocoding

This notebook demonstrates the **SpatiaLite geocoding cache** — a portable,
file-based cache for geocoding results, boundary lookups, and crosswalk mappings.

Also covers the **isochrone** API for drive-time/walk-time service area analysis.

Features:
- Cache-through geocoding (check cache → fall back to Nominatim)
- Bounding-box spatial queries on cached results
- Boundary and crosswalk caching for offline workflows
- Isochrone request building for ORS/Valhalla routing services

In [ ]:
import tempfile
import os

from siege_utilities.geo.geocoding import (
    SpatiaLiteCache,
    concatenate_addresses,
    get_country_name,
    list_countries,
)

## 1. SpatiaLite Cache — Setup

In [ ]:
# Create a temporary cache for this demo
tmp_dir = tempfile.mkdtemp()
cache_path = os.path.join(tmp_dir, "demo_cache.db")
cache = SpatiaLiteCache(db_path=cache_path)

print(f"Cache database: {cache_path}")
print(f"Initial stats: {cache.stats()}")

## 2. Geocode Caching

Store geocoding results so you never pay for the same lookup twice.

In [ ]:
# Cache some geocoding results
addresses = [
    ("1600 Pennsylvania Ave NW, Washington, DC 20500", 38.8977, -77.0365),
    ("350 Fifth Ave, New York, NY 10118", 40.7484, -73.9857),
    ("1 Infinite Loop, Cupertino, CA 95014", 37.3318, -122.0312),
    ("1060 W Addison St, Chicago, IL 60613", 41.9484, -87.6553),
    ("233 S Wacker Dr, Chicago, IL 60606", 41.8789, -87.6359),
]

for addr, lat, lon in addresses:
    addr_hash = cache.put_geocode(addr, lat, lon, source="demo")
    print(f"  Cached: {addr[:40]}... → ({lat}, {lon})")

print(f"\nCache stats: {cache.stats()}")

In [ ]:
# Look up a cached result
result = cache.get_geocode("350 Fifth Ave, New York, NY 10118")
if result:
    print(f"Found: {result['address']}")
    print(f"  Lat: {result['latitude']}, Lon: {result['longitude']}")
    print(f"  WKT: {result['point_wkt']}")
    print(f"  Source: {result['source']}")
    print(f"  Cached at: {result['created_at']}")

# Miss returns None
miss = cache.get_geocode("nonexistent address")
print(f"\nMiss: {miss}")

## 3. Bounding Box Queries

Find all cached geocodes within a geographic rectangle.

In [ ]:
# Find all cached points in the Chicago area
chicago_results = cache.get_geocodes_in_bbox(
    lat_min=41.7, lat_max=42.0,
    lon_min=-87.8, lon_max=-87.5,
)
print(f"Points in Chicago bbox: {len(chicago_results)}")
for r in chicago_results:
    print(f"  {r['address'][:50]}")

# No results in London
london = cache.get_geocodes_in_bbox(51.3, 51.7, -0.5, 0.3)
print(f"\nPoints in London bbox: {len(london)}")

## 4. Boundary Caching

In [ ]:
# Cache boundary lookups for offline use
cache.put_boundary("06037", 2020, point_wkt="POINT(-118.2437 34.0522)", source="tiger")
cache.put_boundary("06037", 2010, point_wkt="POINT(-118.2400 34.0500)", source="tiger")
cache.put_boundary("17031", 2020, point_wkt="POINT(-87.6298 41.8781)", source="tiger")

# Look up
la_2020 = cache.get_boundary("06037", 2020)
print(f"LA County 2020: {la_2020['point_wkt']}")

la_2010 = cache.get_boundary("06037", 2010)
print(f"LA County 2010: {la_2010['point_wkt']}")

print(f"\nCache stats: {cache.stats()}")

## 5. Crosswalk Caching

In [ ]:
# Cache tract crosswalk mappings (2010 → 2020)
cache.put_crosswalk("06037100100", "06037200100", weight=0.75)
cache.put_crosswalk("06037100100", "06037200200", weight=0.25)

# Look up all target tracts for a source
targets = cache.get_crosswalk("06037100100")
print(f"Crosswalk for 06037100100:")
for t in targets:
    print(f"  → {t['target_geoid']} (weight={t['weight']})")

## 6. Isochrone Request Building

Build requests for drive-time/walk-time service areas from routing providers.

In [ ]:
from siege_utilities.geo.isochrones import build_isochrone_request

# Build an ORS isochrone request (15-minute drive from downtown DC)
request = build_isochrone_request(
    latitude=38.8977,
    longitude=-77.0365,
    travel_time_minutes=15,
    provider="openrouteservice",
    profile="driving-car",
)
print(f"Provider: {request['provider']}")
print(f"Method: {request['method']}")
print(f"URL: {request['url']}")
print(f"Params: {request['json']}")

# Build a Valhalla request (30-minute walk)
walk_request = build_isochrone_request(
    latitude=40.7484,
    longitude=-73.9857,
    travel_time_minutes=30,
    provider="valhalla",
    profile="pedestrian",
    base_url="http://valhalla.example.com",
)
print(f"\nValhalla request URL: {walk_request['url']}")

## 7. Cleanup

In [ ]:
print(f"Final cache stats: {cache.stats()}")
cache.close()

# The cache file is portable — copy it to another machine
print(f"Cache file size: {os.path.getsize(cache_path):,} bytes")
print(f"Location: {cache_path}")

## Summary

| Feature | Class/Function | Purpose |
|---------|---------------|----------|
| **Geocode cache** | `SpatiaLiteCache.put/get_geocode()` | Cache Nominatim results |
| **Cache-through** | `get_geocode_or_fetch()` | Auto-fetch on miss |
| **Bbox query** | `get_geocodes_in_bbox()` | Spatial search |
| **Boundary cache** | `put/get_boundary()` | Cache TIGER lookups |
| **Crosswalk cache** | `put/get_crosswalk()` | Cache tract mappings |
| **Isochrone request** | `build_isochrone_request()` | ORS/Valhalla API calls |
| **Isochrone fetch** | `get_isochrone()` | Execute + parse response |

**The cache file is a single SQLite database** — portable across machines,
no server required, spatial index for fast bbox queries.